In [1]:
import torch

In [ ]:
num_samples = 10
n = 4
j = 2
u = 3
positions = torch.tensor([i for i in range(n) if i != j])
values = torch.tensor([v for v in range(n) if v != u])
order_og = torch.rand((num_samples, n - 1))
order = order_og.argsort(dim=1)
print(positions,values)



tensor([0, 1, 3]) tensor([0, 1, 2])


In [7]:
order_og, order

(tensor([[0.9067, 0.6721, 0.3381],
         [0.8165, 0.8730, 0.0402],
         [0.3188, 0.2918, 0.0354],
         [0.8400, 0.7284, 0.4395],
         [0.9229, 0.0504, 0.8371],
         [0.7299, 0.5530, 0.5392],
         [0.6327, 0.7807, 0.8136],
         [0.7933, 0.6786, 0.3888],
         [0.4208, 0.2197, 0.7361],
         [0.2688, 0.8677, 0.3020]]),
 tensor([[2, 1, 0],
         [2, 0, 1],
         [2, 1, 0],
         [2, 1, 0],
         [1, 2, 0],
         [2, 1, 0],
         [0, 1, 2],
         [2, 1, 0],
         [1, 0, 2],
         [0, 2, 1]]))

In [8]:
edges = n * (n - 1) // 2
print(edges)

(torch.rand((num_samples, edges)) < 0.5).long()

6


tensor([[1, 1, 1, 0, 0, 0],
        [0, 1, 1, 1, 0, 1],
        [1, 0, 1, 0, 0, 1],
        [0, 1, 0, 0, 1, 0],
        [0, 1, 0, 1, 1, 1],
        [0, 1, 1, 0, 0, 0],
        [1, 0, 1, 0, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 1],
        [0, 1, 0, 0, 1, 0]])

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

graph = torch.tensor([
    [1, 0, 1, 1, 0, 0],
    [0, 1, 0, 0, 1, 1],
], device=device)

phi = torch.tensor([
    [2, 0, 3, 1],
    [1, 3, 0, 2],
], device=device)


row, column = torch.triu_indices(n, n, offset=1)
print(row, column)
adjacency = torch.zeros((len(graph), n, n), dtype=graph.dtype, device=device)

adjacency[:, row, column] = graph
adjacency[:, column, row] = graph
adjacency

tensor([0, 0, 0, 1, 1, 2]) tensor([1, 2, 3, 2, 3, 3])


tensor([[[0, 1, 0, 1],
         [1, 0, 1, 0],
         [0, 1, 0, 0],
         [1, 0, 0, 0]],

        [[0, 0, 1, 0],
         [0, 0, 0, 1],
         [1, 0, 0, 1],
         [0, 1, 1, 0]]], device='cuda:0')

In [16]:
inverse_phi = torch.argsort(phi, dim=1)
inverse_phi

tensor([[1, 3, 0, 2],
        [2, 0, 3, 1]], device='cuda:0')

In [17]:
batch = torch.arange(len(graph), device=device)[:, None, None]
batch

tensor([[[0]],

        [[1]]], device='cuda:0')

In [20]:
relabeled = adjacency[
    batch,
    inverse_phi[:, :, None],
    inverse_phi[:, None, :],
]
relabeled

tensor([[[0, 0, 1, 1],
         [0, 0, 1, 0],
         [1, 1, 0, 0],
         [1, 0, 0, 0]],

        [[0, 1, 1, 0],
         [1, 0, 0, 0],
         [1, 0, 0, 1],
         [0, 0, 1, 0]]], device='cuda:0')

In [ ]:
relabeled[:, row, column]

tensor([[0, 1, 1, 1, 0, 0],
        [1, 1, 0, 0, 0, 1]], device='cuda:0')